In [ ]:
!pip install -q gradio


In [4]:
import os
import tempfile

import cv2
import gradio as gr
import numpy as np


def make_sample(width=720, height=480):
    # A colourful gradient makes soft and strong edges easy to compare.
    x = np.linspace(0, 1, width, dtype=np.float32)[None, :]
    y = np.linspace(0, 1, height, dtype=np.float32)[:, None]
    image = np.empty((height, width, 3), dtype=np.uint8)
    image[..., 0] = np.clip(35 + 145 * x + 35 * y, 0, 255)
    image[..., 1] = np.clip(35 + 95 * (1 - x) + 90 * y, 0, 255)
    image[..., 2] = np.clip(110 + 100 * (1 - y) + 20 * x, 0, 255)

    cv2.rectangle(image, (55, 65), (285, 250), (245, 245, 250), -1)
    cv2.rectangle(image, (55, 65), (285, 250), (25, 35, 55), 7)
    cv2.line(image, (55, 160), (250, 160), (210, 210, 210), 3, cv2.LINE_AA)
    cv2.circle(image, (505, 155), 92, (245, 185, 45), -1, cv2.LINE_AA)
    cv2.circle(image, (505, 155), 92, (30, 40, 65), 7, cv2.LINE_AA)
    cv2.line(image, (65, 365), (650, 310), (250, 250, 250), 14, cv2.LINE_AA)
    cv2.putText(
        image, "EDGE", (225, 420), cv2.FONT_HERSHEY_DUPLEX,
        2.1, (22, 28, 45), 5, cv2.LINE_AA
    )
    return image


SAMPLE_IMAGE = make_sample()
OUTPUT_PATH = os.path.join(tempfile.gettempdir(), "edge_detector_result.png")


def as_rgb_uint8(image):
    if image is None:
        return SAMPLE_IMAGE.copy()

    image = np.asarray(image)
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    return image


def save_png(rgb_image):
    # Save the same array returned to the preview, so the PNG matches it exactly.
    cv2.imwrite(OUTPUT_PATH, cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR))
    return OUTPUT_PATH


def detect_edges(image, blur_size, low_threshold, high_threshold, overlay):
    original = as_rgb_uint8(image)
    blur_size = int(blur_size)
    low_threshold = int(low_threshold)
    high_threshold = int(high_threshold)

    if low_threshold > high_threshold:
        empty = np.zeros_like(original)
        return (
            original,
            empty,
            "⚠️ Low threshold must be less than or equal to the high threshold.",
            save_png(empty),
        )

    gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
    smoothed = gray if blur_size == 1 else cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
    edges = cv2.Canny(smoothed, low_threshold, high_threshold)
    edge_mask = edges > 0

    if overlay:
        result = original.copy()
        accent = np.array([255, 70, 45], dtype=np.float32)
        result[edge_mask] = (
            0.25 * result[edge_mask].astype(np.float32) + 0.75 * accent
        ).astype(np.uint8)
    else:
        result = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

    count = int(np.count_nonzero(edge_mask))
    share = 100.0 * count / edge_mask.size
    status = f"**Edge pixels:** {share:.2f}% · {count:,} of {edge_mask.size:,} pixels"
    return original, result, status, save_png(result)


initial_original, initial_edges, initial_status, initial_download = detect_edges(
    None, 3, 60, 140, False
)

theme = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="slate",
    neutral_hue="slate",
)

css = """
.gradio-container { max-width: 1180px !important; }
.app-title { margin-bottom: 0.1rem !important; }
.app-note { color: var(--body-text-color-subdued); margin-top: 0 !important; }
.status-card { padding: 0.8rem 1rem; border: 1px solid var(--border-color-primary); border-radius: 12px; }
"""

with gr.Blocks(title="Edge detector") as demo:
    gr.Markdown("# Edge detector", elem_classes=["app-title"])
    gr.Markdown(
        "Upload a JPG or PNG, then tune the blur and Canny thresholds. "
        "With no upload, the built-in sample is used.",
        elem_classes=["app-note"],
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            image_input = gr.Image(
                label="Upload image",
                sources=["upload"],
                type="numpy",
            )
            blur = gr.Slider(
                minimum=1, maximum=7, value=3, step=2, label="Gaussian blur kernel"
            )
            low = gr.Slider(0, 255, value=60, step=1, label="Low threshold")
            high = gr.Slider(0, 255, value=140, step=1, label="High threshold")
            overlay = gr.Checkbox(value=False, label="Overlay coloured edges")
            stats = gr.Markdown(initial_status, elem_classes=["status-card"])
            download = gr.DownloadButton(
                "Download edge image (PNG)", value=initial_download, variant="primary"
            )

        with gr.Column(scale=2, min_width=420):
            with gr.Row():
                original_output = gr.Image(
                    value=initial_original, label="Original", interactive=False
                )
                edge_output = gr.Image(
                    value=initial_edges, label="Edges", interactive=False
                )

    inputs = [image_input, blur, low, high, overlay]
    outputs = [original_output, edge_output, stats, download]
    image_input.change(detect_edges, inputs=inputs, outputs=outputs)
    for control in [blur, low, high, overlay]:
        control.input(detect_edges, inputs=inputs, outputs=outputs)

demo.queue().launch(theme=theme, css=css, share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ce5b67cbc4d672def.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
